# Discrete Adjoint via AD (JAX + Diffrax) — Two-Block Spring-Slider Inversion

This notebook re-implements the two-block forward model in JAX and obtains the gradient by **backpropagating directly through the adaptive ODE stepper** using Diffrax's `RecursiveCheckpointAdjoint`. This is the *discrete* adjoint of the actual time-stepping algorithm — exact for the discretised problem by construction.

**Why this approach.** The continuous adjoint of the two-block system became dual-inconsistent during fast slip events: the adjoint integrates against forward-grid-interpolated Jacobians whose denominator $\tau_V + \eta$ pinches near rupture, producing spurious blowup (Alexe & Sandu, *JCAM* 2009). Forward sensitivity (the previous fix) sidesteps the issue but scales linearly in the number of parameters. Discrete-adjoint-via-AD also sidesteps it — and scales independently of the parameter count, which matters once we want to invert for more than 4 parameters.

**Topology and physics** are unchanged from the existing two-block notebook:
- $\tau_i(V_i,\psi_i) + \eta V_i + (k_0{+}k_{12})u_i - k_{12}u_j = \tau_{0,i} + k_0 V_{\rm bg} t$ (force balance, both blocks)
- $\dot u_i = V_i$, $\dot\psi_i = G(V_i,\psi_i) = \frac{b V_0}{d_c}e^{-(\psi_i-f_0)/b} - \frac{b V_i}{d_c}$ (aging law)
- $\tau(V,\psi) = N a\,\text{arcsinh}(V/(2V_0)\,e^{\psi/a})$ (regularised RS)

**Key implementation choices:**
1. **Algebraic constraint** $\tau + \eta V = \text{rhs}$ is solved via `optimistix.Newton` in $\log V$ inside the ODE RHS. Optimistix is a differentiable implicit-layer library — AD propagates through the root-find via the implicit function theorem, so backprop just works.
2. **Time stepping** uses `diffrax.Tsit5` (5th-order explicit RK with PI step-size control). Saving at the fixed $t_{\rm ref}$ grid happens via `SaveAt(ts=t_ref)` — no separate post-hoc interpolation.
3. **Frozen-IC convention** is preserved: $u_{i,0}$, $\psi_{i,0}$, $\tau_{0,i}$ are computed once from the initial guess and held constant across iterates.


In [1]:
%pip install -q --upgrade jax jaxlib diffrax optimistix equinox

ERROR: Could not find a version that satisfies the requirement jaxlib (from versions: none)
ERROR: No matching distribution found for jaxlib
Note: you may need to restart the kernel to use updated packages.


In [ ]:
%load_ext autoreload
%autoreload 2

import jax
jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import diffrax
import optimistix as optx
from scipy.optimize import minimize, Bounds

print(f"JAX:        {jax.__version__}")
print(f"Diffrax:    {diffrax.__version__}")
print(f"Optimistix: {optx.__version__}")
print(f"x64 enabled: {jax.config.read('jax_enable_x64')}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


ModuleNotFoundError: No module named 'diffrax'

## Parameters and Initial Conditions

Identical to the existing two-block notebook so results are directly comparable.


In [ ]:
# Parameters - two-block coupled system (Abe & Kato 2013 topology)
M = {}
M['f0']   = 0.6
M['V0']   = 1e-6
M['a1']   = 0.010
M['a2']   = 0.011
M['b']    = 0.012
M['dc']   = 1e-4
M['N']    = 50.0
M['eta']  = 2.7 * 3.5 / 2.0
M['V_bg'] = 1e-9

k_crit1   = M['N'] * (M['b'] - M['a1']) / M['dc']
M['k0']   = abs(0.6  * k_crit1)
M['k12']  = abs(0.35 * k_crit1)

V1_init = 1.0e-12
V2_init = 1.0e-12

T = 1.9e7      # s
print(f"a1={M['a1']},  a2={M['a2']},  b={M['b']}")
print(f"k0 ={M['k0']:.3e},  k12={M['k12']:.3e}  (MPa/m)")
print(f"T = {T:.2e} s  ({T/86400:.0f} days)")

In [ ]:
# Frozen IC + tau0_i (computed once from these params; held constant across iterates)
from friction_derivs import setup_initial_conditions_2block

u1_0, psi1_0, _, u2_0, psi2_0, _ = setup_initial_conditions_2block(
    M, V1_init=V1_init, V2_init=V2_init)
tau0_1 = M['tau0_1']
tau0_2 = M['tau0_2']

print(f"u1_0={u1_0},  psi1_0={psi1_0:.4f},  tau0_1={tau0_1:.6f}")
print(f"u2_0={u2_0},  psi2_0={psi2_0:.4f},  tau0_2={tau0_2:.6f}")

## JAX Physics Primitives

Same equations as `friction_derivs.py`, rewritten in `jnp` so AD can trace them. The constants `f0, V0, b, dc, N, eta, V_bg` are captured by closure as Python floats (they don't appear in the parameter vector and have no gradient).


In [ ]:
f0   = M['f0']
V0   = M['V0']
b    = M['b']
dc   = M['dc']
N    = M['N']
eta  = M['eta']
V_bg = M['V_bg']

def tau_fn(V, psi, a):
    # tau = N*a*arcsinh(V/(2V0)*exp(psi/a))
    xi = V / (2.0 * V0) * jnp.exp(psi / a)
    return N * a * jnp.arcsinh(xi)

def G_fn(V, psi):
    # aging-law dpsi/dt
    return b * V0 / dc * jnp.exp(-(psi - f0) / b) - b * V / dc

## Differentiable Force-Balance Solver via Optimistix

The algebraic constraint $\tau(V,\psi) + \eta V = \text{rhs}$ is monotone in $V$ (both terms are increasing for $V>0$). We solve it via Newton's method on $\log V$ — log-space keeps the iteration stable across the 12+ orders of magnitude $V$ traverses during a rupture.

`optimistix.root_find` differentiates through the root-find via the implicit function theorem, so backpropagation through this step happens automatically.


In [ ]:
def _V_residual_logV(logV, args):
    psi, a, rhs = args
    V = jnp.exp(logV)
    return tau_fn(V, psi, a) + eta * V - rhs

_V_solver = optx.Newton(rtol=1e-12, atol=1e-14)

def solve_V(psi, a, rhs):
    # Return V > 0 satisfying tau(V,psi) + eta*V = rhs
    # Upper bound is rhs/eta (friction = 0); start mid-bracket in log space.
    logV0 = jnp.log(jnp.maximum(rhs, 1e-30) / (2.0 * eta))
    sol = optx.root_find(_V_residual_logV, _V_solver, logV0,
                          args=(psi, a, rhs),
                          max_steps=200, throw=False)
    return jnp.exp(sol.value)

# Quick sanity check: solve at the initial state
rhs_test = tau0_1
V_test = float(solve_V(jnp.asarray(psi1_0), jnp.asarray(M['a1']), jnp.asarray(rhs_test)))
print(f"solve_V at IC: V1 = {V_test:.4e}  (V1_init = {V1_init:.4e})")
assert abs(V_test - V1_init) / V1_init < 1e-6, "Force-balance solver disagrees with IC"
print("OK")

## ODE Vector Field (Two-Block, $V$ Eliminated)

The DAE collapses to a 4-D ODE in $(u_1, \psi_1, u_2, \psi_2)$ once we solve the algebraic $V_i$ constraint inside the RHS. Diffrax sees only the ODE.


In [ ]:
def vector_field(t, y, args):
    a1, a2, k0, k12 = args
    u1, psi1, u2, psi2 = y[0], y[1], y[2], y[3]

    rhs1 = tau0_1 + k0 * V_bg * t - (k0 + k12) * u1 + k12 * u2
    rhs2 = tau0_2 + k0 * V_bg * t - (k0 + k12) * u2 + k12 * u1

    V1 = solve_V(psi1, a1, rhs1)
    V2 = solve_V(psi2, a2, rhs2)

    return jnp.stack([V1, G_fn(V1, psi1), V2, G_fn(V2, psi2)])

## Forward Solve with `RecursiveCheckpointAdjoint`

Diffrax's `RecursiveCheckpointAdjoint` performs reverse-mode AD through the actual time-stepping algorithm with logarithmic checkpointing. The official Diffrax docs explicitly recommend it over `BacksolveAdjoint` (the continuous adjoint) for accuracy on stiff or near-stiff systems — which is exactly the failure mode we hit before.

We use `Tsit5` (5th-order explicit RK with PI step control) and tight tolerances. `SaveAt(ts=t_ref)` returns the state on the fixed reference grid that $J$ uses, so no post-hoc interpolation is needed.


In [ ]:
SOLVER         = diffrax.Tsit5()
RTOL           = 1e-8
ATOL           = 1e-10
MAX_STEPS      = 200_000

_controller = diffrax.PIDController(rtol=RTOL, atol=ATOL)
_term       = diffrax.ODETerm(vector_field)
_adjoint    = diffrax.RecursiveCheckpointAdjoint()

def forward_solve_jax(params_vec, t_save):
    # Returns ys: (len(t_save), 4) array of (u1, psi1, u2, psi2) at t_save.
    a1, a2, k0, k12 = params_vec[0], params_vec[1], params_vec[2], params_vec[3]
    y0 = jnp.array([u1_0, psi1_0, u2_0, psi2_0], dtype=jnp.float64)

    sol = diffrax.diffeqsolve(
        _term, SOLVER,
        t0=0.0, t1=float(t_save[-1]), dt0=1.0,
        y0=y0,
        args=(a1, a2, k0, k12),
        saveat=diffrax.SaveAt(ts=t_save),
        stepsize_controller=_controller,
        adjoint=_adjoint,
        max_steps=MAX_STEPS,
    )
    return sol.ys

# Dry-run at true params to compile + verify
p_true = jnp.array([M['a1'], M['a2'], M['k0'], M['k12']], dtype=jnp.float64)
t_dense = jnp.linspace(0.0, T, 2000)
import time
t0 = time.time(); ys = forward_solve_jax(p_true, t_dense); ys.block_until_ready()
print(f"First (compile+run) forward solve: {time.time()-t0:.2f} s")
t0 = time.time(); ys = forward_solve_jax(p_true, t_dense); ys.block_until_ready()
print(f"Second (cached)   forward solve: {time.time()-t0:.2f} s")
print(f"u1(T) = {float(ys[-1,0]):.4e},  u2(T) = {float(ys[-1,2]):.4e}")

## Plot Forward Solution (sanity-check against the existing numpy solver)


In [ ]:
from adapt_fwd_solve import forward_solve_adaptive_2block

fwd_ref = forward_solve_adaptive_2block(dict(M), T, u1_0, psi1_0, u2_0, psi2_0,
                                         V1_init=V1_init, V2_init=V2_init)

fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(fwd_ref['t'], fwd_ref['u1'], 'k--', lw=1.5, label='numpy (reference)')
axes[0].plot(t_dense,      ys[:, 0],      'C0',  lw=1.2, label='JAX/Diffrax')
axes[0].set_ylabel('$u_1$ (m)'); axes[0].legend(fontsize=8); axes[0].grid(True, ls=':')

axes[1].semilogy(fwd_ref['t'], fwd_ref['V1'], 'k--', lw=1.5, label='numpy $V_1$')
# Recover V from the JAX solution at the saved times
V1_jax = np.array([float(solve_V(jnp.asarray(p), jnp.asarray(M['a1']),
                                  jnp.asarray(tau0_1 + M['k0']*V_bg*t - (M['k0']+M['k12'])*u + M['k12']*u2)))
                   for t, u, p, u2 in zip(t_dense, ys[:,0], ys[:,1], ys[:,2])])
axes[1].semilogy(t_dense, V1_jax, 'C0', lw=1.2, label='JAX $V_1$')
axes[1].set_ylabel('$V_1$ (m/s)'); axes[1].set_xlabel('Time (s)')
axes[1].legend(fontsize=8); axes[1].grid(True, ls=':', which='both')
plt.tight_layout(); plt.show()

rel_u1 = np.max(np.abs(np.interp(t_dense, fwd_ref['t'], fwd_ref['u1']) - ys[:, 0])) / np.max(np.abs(fwd_ref['u1']))
rel_u2 = np.max(np.abs(np.interp(t_dense, fwd_ref['t'], fwd_ref['u2']) - ys[:, 2])) / np.max(np.abs(fwd_ref['u2']))
print(f"max |u1_JAX - u1_numpy| / max|u1| = {rel_u1:.2e}")
print(f"max |u2_JAX - u2_numpy| / max|u2| = {rel_u2:.2e}")

## Synthetic Observations and Objective

Generate observations at the true parameters using the JAX forward (so the test is end-to-end self-consistent in the AD framework). Build the fixed reference grid + smoothing matrix.


In [ ]:
from friction_derivs import make_smoothing_matrix

sigma_smooth = 0.0 * T
t_ref_np = np.linspace(0.0, T, 1000)
S_np     = make_smoothing_matrix(t_ref_np, sigma_smooth)

t_ref = jnp.asarray(t_ref_np)
S     = jnp.asarray(S_np)

# True observations on t_ref
ys_obs = forward_solve_jax(p_true, t_ref)
u1_obs = ys_obs[:, 0]
u2_obs = ys_obs[:, 2]
Su1_obs = S @ u1_obs
Su2_obs = S @ u2_obs

print(f"u1_obs(T) = {float(u1_obs[-1]):.4e},  u2_obs(T) = {float(u2_obs[-1]):.4e}")
print(f"t_ref: {len(t_ref_np)} points,  S: {S.shape}")

In [ ]:
def J_fn(p_vec):
    ys = forward_solve_jax(p_vec, t_ref)
    u1 = ys[:, 0]
    u2 = ys[:, 2]
    r1 = S @ u1 - Su1_obs
    r2 = S @ u2 - Su2_obs
    return 0.5 * jnp.trapezoid(r1**2 + r2**2, t_ref)

J_and_grad = jax.jit(jax.value_and_grad(J_fn))

# Sanity: J at true params should be ~ 0 (modulo solver tolerance)
J0, g0 = J_and_grad(p_true)
J0.block_until_ready()
print(f"J(p_true) = {float(J0):.4e}   (should be ~0)")
print(f"grad J(p_true) = {np.asarray(g0)}   (should be ~0)")

## Gradient Validation: Discrete-Adjoint AD vs Finite Differences

The acid test. At a perturbed parameter point (away from the minimum so gradients are nonzero), compare the AD gradient against centred finite differences. Relative agreement to within $\lesssim 10^{-4}$ is the success criterion — anything looser means either the AD or the FD is unreliable.


In [ ]:
PERTURB_IDX = 2     # 0=a1, 1=a2, 2=k0, 3=k12
PERTURB_F   = 1.1

p_test = p_true.at[PERTURB_IDX].set(p_true[PERTURB_IDX] * PERTURB_F)
J_test, g_ad = J_and_grad(p_test)
J_test.block_until_ready()
g_ad_np = np.asarray(g_ad)
print(f"At p_test (perturbed param idx {PERTURB_IDX}, factor {PERTURB_F}):")
print(f"  J = {float(J_test):.6e}")
print(f"  AD grad = {g_ad_np}")

print("\nComputing centred FD (eps = 1e-6 * |p|, two J-evals per param) ...")
g_fd = np.zeros(4)
for i in range(4):
    eps = float(p_test[i]) * 1e-6
    pp = p_test.at[i].set(p_test[i] + eps)
    pm = p_test.at[i].set(p_test[i] - eps)
    Jp, _ = J_and_grad(pp)
    Jm, _ = J_and_grad(pm)
    g_fd[i] = (float(Jp) - float(Jm)) / (2.0 * eps)

print(f"\n{'param':<6} {'FD':>16} {'AD':>16} {'|AD-FD|/|FD|':>16}")
print('-' * 60)
for i, name in enumerate(['a1', 'a2', 'k0', 'k12']):
    rel = abs(g_ad_np[i] - g_fd[i]) / (abs(g_fd[i]) + 1e-30)
    print(f"{name:<6} {g_fd[i]:+16.6e} {g_ad_np[i]:+16.6e} {rel:16.2e}")

In [ ]:
# Bar-chart comparison
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, i, name in zip(axes, range(4), ['a1', 'a2', 'k0', 'k12']):
    vals = [g_fd[i], g_ad_np[i]]
    bars = ax.bar(['FD', 'AD'], vals, color=['C0', 'C2'], alpha=0.85, edgecolor='k')
    ax.axhline(g_fd[i], color='k', ls='--', lw=0.7)
    ax.set_title(f'dJ/d{name}')
    ax.grid(True, ls=':', axis='y')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v, f'{v:.2e}',
                ha='center', va='bottom' if v >= 0 else 'top', fontsize=8)
fig.suptitle(f'Discrete-adjoint AD vs FD  (perturbed idx {PERTURB_IDX} by {PERTURB_F}x)')
plt.tight_layout(); plt.show()

## Inversion via `scipy.optimize.minimize` with the AD Gradient

Same setup as the existing forward-sensitivity notebook: invert for $\{a_1, a_2, k_0, k_{12}\}$, parameters normalised by their initial guesses (so all components are $O(1)$ inside the optimiser), `trust-constr` with bounds.

The only thing that changed under the hood is the gradient source: now `jax.value_and_grad(J)` via discrete backprop through Diffrax.


In [ ]:
INVERT_PARAMS = ['a1', 'a2', 'k0', 'k12']
PARAM_IDX = {'a1': 0, 'a2': 1, 'k0': 2, 'k12': 3}

_p_true_d = {p: M[p] for p in INVERT_PARAMS}
_p_init_d = {
    'a1':  M['a1']  * 1.1,
    'a2':  M['a2']  * 1.1,
    'k0':  M['k0']  * 0.9,
    'k12': M['k12'] * 0.9,
}
_p_bnds_d = {p: (_p_true_d[p] * 0.5, _p_true_d[p] * 1.5) for p in INVERT_PARAMS}

scales = np.array([_p_init_d[p] for p in INVERT_PARAMS])
x0     = np.array([_p_init_d[p] for p in INVERT_PARAMS]) / scales
lb     = np.array([_p_bnds_d[p][0] for p in INVERT_PARAMS]) / scales
ub     = np.array([_p_bnds_d[p][1] for p in INVERT_PARAMS]) / scales

J_hist, x_hist, grad_hist = [], [], []

def fun_and_grad(x_norm):
    p_full = np.array([M[p] for p in ['a1', 'a2', 'k0', 'k12']])
    for i, name in enumerate(INVERT_PARAMS):
        p_full[PARAM_IDX[name]] = float(x_norm[i] * scales[i])
    p_vec = jnp.asarray(p_full)
    Jv, gv = J_and_grad(p_vec)
    Jv.block_until_ready()
    g_full = np.asarray(gv)
    g_active = np.array([g_full[PARAM_IDX[name]] for name in INVERT_PARAMS])

    J_hist.append(float(Jv))
    x_hist.append(x_norm * scales)
    grad_hist.append(g_active.copy())

    it = len(J_hist)
    pstr = ',  '.join(
        f"{name}={float(x_norm[i]*scales[i]):.6g} "
        f"({abs(float(x_norm[i]*scales[i])-_p_true_d[name])/_p_true_d[name]*100:.3f}% err)"
        for i, name in enumerate(INVERT_PARAMS))
    print(f"eval {it:3d}: J={float(Jv):.4e},  {pstr}")
    return float(Jv), g_active * scales

print(f"Inverting for: {INVERT_PARAMS}")
print(f"Initial: {[_p_init_d[p] for p in INVERT_PARAMS]}")
print(f"True:    {[_p_true_d[p] for p in INVERT_PARAMS]}")

In [ ]:
J_hist.clear(); x_hist.clear(); grad_hist.clear()

result = minimize(
    fun=fun_and_grad,
    x0=x0,
    method='trust-constr',
    jac=True,
    bounds=Bounds(lb, ub),
    options={'maxiter': 40, 'gtol': 1e-6, 'initial_tr_radius': 0.005},
)

x_final = result.x * scales
print(f"\n{result.message}")
for i, name in enumerate(INVERT_PARAMS):
    tv = _p_true_d[name]; fv = x_final[i]
    print(f"  {name}: {fv:.6g}  (true={tv:.6g},  err={abs(fv-tv)/tv*100:.4f}%)")
print(f"Total evaluations: {len(J_hist)}")

## Convergence Plots


In [ ]:
n_evals = len(J_hist)
iters = np.arange(1, n_evals + 1)
x_arr = np.array(x_hist)

fig, axes = plt.subplots(1 + len(INVERT_PARAMS), 1,
                          figsize=(8, 3 * (1 + len(INVERT_PARAMS))), sharex=True)
axes[0].semilogy(iters, J_hist, 'o-', ms=4)
axes[0].set_ylabel('$J$')
axes[0].set_title('Discrete-adjoint-via-AD inversion convergence')
axes[0].grid(True, ls=':')

for i, name in enumerate(INVERT_PARAMS):
    ax = axes[1 + i]
    ax.plot(iters, x_arr[:, i], 'o-', ms=4, label=f'${name}$ (estimated)')
    ax.axhline(_p_true_d[name], color='k', ls='--', lw=1.0,
               label=f'${name}_{{\\rm true}}$ = {_p_true_d[name]:.5g}')
    ax.set_ylabel(f'${name}$')
    ax.legend(fontsize=8); ax.grid(True, ls=':')

axes[-1].set_xlabel('Function evaluation')
plt.tight_layout(); plt.show()

print('\nFinal parameter estimates:')
for i, name in enumerate(INVERT_PARAMS):
    print(f"  {name}: {x_final[i]:.6g}  (true={_p_true_d[name]:.6g},  "
          f"err={abs(x_final[i]-_p_true_d[name])/_p_true_d[name]*100:.4f}%)")